# Signature-analysis SLURM dispatch

Mirrors `hvg_dispatch.ipynb`: builds the `signature_run.py --config <name>` command for a
dataset and submits it via `simple_slurm`. CPU-only (no `gres`) -- the signature fits are
plain OLS/L1 over cached factor stores, no GPU needed. Run only the config cell(s) you
want; `signature_run.py` itself skips any work already cached unless `--overwrite` is
passed.

In [4]:
import sys
from pathlib import Path
from datetime import timedelta

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from simple_slurm import Slurm

PYTHON = '/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python'
SCRIPT = str(_root / 'metab_processing' / 'LinearRegression' / 'signature_run.py')
LOG_DIR = Path('/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/sig')
LOG_DIR.mkdir(parents=True, exist_ok=True)

# VERIFY partition+QOS with: sacctmgr -p show assoc user=$USER
# These are allocation-specific GUESSES, not confirmed -- check before relying on them.
SLURM_BY_CONFIG = {
    'cervical': ('savio3_bigmem', 'savio_normal'),
    'breast': ('savio3_htc', 'savio_normal'),
    'colon': ('savio3_htc', 'savio_normal'),
}

## cervical

In [ ]:
name = 'cervical'
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SCRIPT} --config {name}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=32, time=timedelta(hours=48), ignore_pbs=True,
              job_name=f'sig_{name}', output=str(LOG_DIR / f'sig_{name}_%j.log'))
slurm.sbatch(cmd)

/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/LinearRegression/signature_run.py --config cervical
Submitted batch job 41051117



41051117

## breast

In [2]:
name = 'breast'
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SCRIPT} --config {name}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=8, time=timedelta(hours=24), ignore_pbs=True,
              job_name=f'sig_{name}', output=str(LOG_DIR / f'sig_{name}_%j.log'))
slurm.sbatch(cmd)

/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/LinearRegression/signature_run.py --config breast
Submitted batch job 41051106



41051106

## colon

In [4]:
name = 'colon'
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SCRIPT} --config {name}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=8, time=timedelta(hours=24), ignore_pbs=True,
              job_name=f'sig_{name}', output=str(LOG_DIR / f'sig_{name}_%j.log'))
slurm.sbatch(cmd)

/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/LinearRegression/signature_run.py --config colon
Submitted batch job 40616596



40616596

# Single slice (manual) — one job over a chosen sub-slice

The simplest form: **cell 1** picks a slice of the target array (e.g. the **last 75%**,
`targets[len(targets)//4:]`), **cell 2** dispatches that slice as one `--targets` job. No
fan-out — just one job on the targets you select. (For splitting across *several* concurrent
jobs, use the "Parallel fan-out" section below instead.)

In [ ]:
# 1) Pick the slice of targets to run. Edit `sel`.
from metab_processing.LinearRegression.signature_batch import all_targets
from metab_processing.LinearRegression.signature_configs import CONFIGS

name = 'cervical'                      # 'cervical' or 'breast'
targets = list(all_targets(CONFIGS[name]))

sel = targets[len(targets) // 4:]      # <-- the LAST 75% (drops the first quarter)
# other handy slices:
#   targets[:len(targets) // 4]        # first 25%
#   targets[5:10]                      # an explicit index range
#   [t for t in targets if 'HYPOXIA' in t]   # pick by name

print(f'{name}: {len(targets)} targets total; selected {len(sel)}:')
for t in sel:
    print('  ', t)

In [ ]:
# 2) Dispatch the selected slice (`sel`) as ONE job.
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SCRIPT} --config {name} --targets {",".join(sel)}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=32, time=timedelta(hours=48), ignore_pbs=True,   # cervical; breast uses 8 / 24h
              job_name=f'sig_{name}', output=str(LOG_DIR / f'sig_{name}_%j.log'))
slurm.sbatch(cmd)

# Parallel fan-out (split one dataset across concurrent jobs)

Split **one** dataset's target list across `n_jobs` concurrent SLURM jobs instead of a
single job that walks every target. Each job gets a contiguous slice via `--targets`; the
per-`(target, run-spec)` cache keying means the slices write **disjoint** files into the
shared cache dir, so they're safe to run at once. Use for **cervical** or **breast** when
the single-job run is too slow. The viewer notebooks are unchanged -- they read the merged
cache. (Leave `--targets` off, i.e. the per-dataset cells above, for a single-job run.)

In [ ]:
# --- Parallel fan-out: split ONE dataset's targets across N concurrent jobs ---
# Each job runs `signature_run.py --config <name> --targets <slice>`. Results are cached
# per (target, run-spec), so the slices write DISJOINT files into the same cache dir --
# safe concurrent writes. (_membership.json + neighborhood factors are rebuilt in full by
# every job; only the per-target fits are split.) The viewer notebook reads the merged
# cache unchanged. Set `name` + `n_jobs`, run this cell.
import numpy as np
from metab_processing.LinearRegression.signature_batch import all_targets
from metab_processing.LinearRegression.signature_configs import CONFIGS

name = 'cervical'        # 'cervical' or 'breast'
n_jobs = 4               # how many concurrent jobs to split this dataset's targets across

# (cpus_per_task, wall-hours) per dataset -- matches the single-job cells above. Each slice
# does fewer targets than the whole, so these are safe (conservative) upper bounds.
RES_BY_CONFIG = {'cervical': (32, 48), 'breast': (8, 24), 'colon': (8, 24)}

partition, qos = SLURM_BY_CONFIG[name]
cpus, hours = RES_BY_CONFIG[name]
targets = list(all_targets(CONFIGS[name]))
slices = [list(s) for s in np.array_split(targets, n_jobs) if len(s)]   # contiguous chunks
print(f'{name}: {len(targets)} targets -> {len(slices)} job(s)')
for i, chunk in enumerate(slices):
    cmd = f'{PYTHON} {SCRIPT} --config {name} --targets {",".join(chunk)}'
    print(f'[job {i}] {len(chunk)} targets: {cmd}')
    slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
                  cpus_per_task=cpus, time=timedelta(hours=hours), ignore_pbs=True,
                  job_name=f'sig_{name}_{i}',
                  output=str(LOG_DIR / f'sig_{name}_{i}_%j.log'))
    slurm.sbatch(cmd)

# Penalty-sweep dispatch

`signature_sweep_run.py --config <name> --method <l1|ridge|...>` computes + caches the
penalty sweep (full + k subsamples) for one dataset+method. Penalties are per-method in the
config; the sweep cache is incremental (re-running only computes new penalties).

In [ ]:
SWEEP_SCRIPT = str(_root / 'metab_processing' / 'LinearRegression' / 'signature_sweep_run.py')

## sweep: cervical

In [ ]:
name, method = 'cervical', 'l1'     # method in cfg.sweep_penalties (e.g. 'l1' or 'ridge')
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SWEEP_SCRIPT} --config {name} --method {method}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=32, time=timedelta(hours=24), ignore_pbs=True,
              job_name=f'sweep_{name}_{method}', output=str(LOG_DIR / f'sweep_{name}_{method}_%j.log'))
slurm.sbatch(cmd)

## sweep: breast

In [ ]:
name, method = 'breast', 'l1'     # method in cfg.sweep_penalties (e.g. 'l1' or 'ridge')
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SWEEP_SCRIPT} --config {name} --method {method}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=8, time=timedelta(hours=24), ignore_pbs=True,
              job_name=f'sweep_{name}_{method}', output=str(LOG_DIR / f'sweep_{name}_{method}_%j.log'))
slurm.sbatch(cmd)

## sweep: colon

In [ ]:
name, method = 'colon', 'l1'     # method in cfg.sweep_penalties (e.g. 'l1' or 'ridge')
partition, qos = SLURM_BY_CONFIG[name]
cmd = f'{PYTHON} {SWEEP_SCRIPT} --config {name} --method {method}'
print(cmd)
slurm = Slurm(account='fc_wagnerlabfca', partition=partition, qos=qos,
              cpus_per_task=8, time=timedelta(hours=24), ignore_pbs=True,
              job_name=f'sweep_{name}_{method}', output=str(LOG_DIR / f'sweep_{name}_{method}_%j.log'))
slurm.sbatch(cmd)